# 12 -- ORCA: Capabilities & Practical Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/12_orca_tutorial.ipynb)

## Learning Objectives
- Write valid ORCA input files for single-point, optimization, and frequency jobs
- Understand the `!` keyword line and `%` block syntax
- Use the RIJCOSX approximation to accelerate hybrid-DFT calculations
- Set up DLPNO-CCSD(T) for benchmark-quality energetics
- Parse ORCA output files with Python and cclib
- Choose appropriate methods, basis sets, and auxiliary basis sets
- Recognize common ORCA error messages and their solutions

## 1. ORCA Overview

**ORCA** (developed by Frank Neese, MPI Mulheim) is a free-for-academic production
quantum chemistry code. Download from https://orcaforum.kofo.mpg.de

| Area | Methods |
|------|--------|
| DFT | All major XC functionals; hybrid, double-hybrid |
| Post-HF | MP2, CCSD(T), DLPNO-CCSD(T) |
| Multireference | CASSCF, NEVPT2, MRCI |
| Spectroscopy | EPR, Mossbauer, X-ray, NMR |
| Relativistic | ZORA, DKH, CPCM solvation |

## 2. Input File Syntax

An ORCA input file (`job.inp`) consists of:
1. **`!` keyword line** -- method, basis set, and job-type keywords
2. **`%` blocks** -- detailed options (optional)
3. **`* xyz` coordinate block** -- molecular geometry

### 2.1 Minimal Single-Point

```
! B3LYP def2-SVP TightSCF

* xyz 0 1
  O   0.000   0.000   0.117
  H   0.000   0.757  -0.469
  H   0.000  -0.757  -0.469
*
```

### 2.2 Geometry Optimization + Frequencies

```
! B3LYP def2-TZVP TightSCF Opt Freq

%maxcore 4000
%pal nprocs 8 end

* xyz 0 1
  O   0.000   0.000   0.117
  H   0.000   0.757  -0.469
  H   0.000  -0.757  -0.469
*
```

### 2.3 RIJCOSX for Fast Hybrid DFT

The **RIJCOSX** approximation (RI-J for Coulomb + COSX for exchange) reduces
the cost of hybrid functionals from O(N^4) to ~O(N^3):

```
! B3LYP def2-TZVP def2/J RIJCOSX TightSCF Opt
```

Always pair with a matching auxiliary basis: `def2/J` for Coulomb fitting;
`def2-TZVP/C` for correlation (MP2/CC).

## 3. DLPNO-CCSD(T): Gold-Standard Energetics

**DLPNO-CCSD(T)** (Domain-based Local Pair Natural Orbital) achieves near-canonical
CCSD(T) accuracy at a fraction of the cost (O(N^3) vs O(N^7)):

```
! DLPNO-CCSD(T) def2-TZVP def2-TZVP/C TightSCF TightPNO

%maxcore 8000
%pal nprocs 16 end

* xyz 0 1
  ...
*
```

**Typical accuracy**: Delta-H errors < 1 kcal/mol vs canonical CCSD(T).

**TightPNO thresholds** (recommended for benchmarks):
- `TCutPNO = 1e-7` (default Normal = 3.33e-7)
- `TCutPairs = 1e-5`
- `TCutMKN = 1e-3`

In [1]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT -- Notebook 12: ORCA Tutorial
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================

# ------------------------------------------------------------------
# Writing ORCA Input Files with Python
# ------------------------------------------------------------------

def write_orca_input(filename, method, basis, charge, mult, atoms,
                     extra_keywords='', extra_blocks=''):
    '''Write a minimal ORCA .inp file.'''
    coord_block = '\n'.join(
        f'  {sym:2s}  {x:10.5f}  {y:10.5f}  {z:10.5f}'
        for sym, x, y, z in atoms
    )
    content = f'! {method} {basis} TightSCF {extra_keywords}\n'
    if extra_blocks:
        content += extra_blocks + '\n'
    content += f'\n* xyz {charge} {mult}\n{coord_block}\n*\n'
    with open(filename, 'w') as f:
        f.write(content)
    return content

water_atoms = [
    ('O',  0.000,  0.000,  0.117),
    ('H',  0.000,  0.757, -0.469),
    ('H',  0.000, -0.757, -0.469),
]

inp_sp = write_orca_input(
    '/tmp/water_sp.inp',
    method='B3LYP', basis='def2-TZVP',
    charge=0, mult=1, atoms=water_atoms,
    extra_keywords='def2/J RIJCOSX'
)
print('=== Water Single-Point (B3LYP/def2-TZVP/RIJCOSX) ===')
print(inp_sp)

# Example: Fe(CO)5 optimization
feco5_atoms = [
    ('Fe',  0.000,  0.000,  0.000),
    ('C',   0.000,  0.000,  1.807), ('C',  0.000,  0.000, -1.807),
    ('C',   1.807,  0.000,  0.000), ('C', -1.807,  0.000,  0.000),
    ('C',   0.000,  1.807,  0.000),
    ('O',   0.000,  0.000,  2.975), ('O',  0.000,  0.000, -2.975),
    ('O',   2.975,  0.000,  0.000), ('O', -2.975,  0.000,  0.000),
    ('O',   0.000,  2.975,  0.000),
]
inp_opt = write_orca_input(
    '/tmp/feco5_opt.inp',
    method='BP86', basis='def2-TZVP',
    charge=0, mult=1, atoms=feco5_atoms,
    extra_keywords='def2/J RI Opt',
    extra_blocks='%maxcore 4000\n%pal nprocs 4 end'
)
print('=== Fe(CO)5 Optimization (BP86/def2-TZVP/RI) ===')
print(inp_opt)

=== Water Single-Point (B3LYP/def2-TZVP/RIJCOSX) ===
! B3LYP def2-TZVP TightSCF def2/J RIJCOSX

* xyz 0 1
  O      0.00000     0.00000     0.11700
  H      0.00000     0.75700    -0.46900
  H      0.00000    -0.75700    -0.46900
*

=== Fe(CO)5 Optimization (BP86/def2-TZVP/RI) ===
! BP86 def2-TZVP TightSCF def2/J RI Opt
%maxcore 4000
%pal nprocs 4 end

* xyz 0 1
  Fe     0.00000     0.00000     0.00000
  C      0.00000     0.00000     1.80700
  C      0.00000     0.00000    -1.80700
  C      1.80700     0.00000     0.00000
  C     -1.80700     0.00000     0.00000
  C      0.00000     1.80700     0.00000
  O      0.00000     0.00000     2.97500
  O      0.00000     0.00000    -2.97500
  O      2.97500     0.00000     0.00000
  O     -2.97500     0.00000     0.00000
  O      0.00000     2.97500     0.00000
*



In [1]:
# =============================================================================
# Minimal Molecular Viewer (py3Dmol)
# =============================================================================

import py3Dmol

def view_structure(filename, style='ballstick', width=500, height=400):
    """
    Minimal, clean py3Dmol viewer for XYZ or PDB files.
    Automatically detects format from file extension.
    """
    ext = filename.split('.')[-1].lower()
    if ext not in ('xyz', 'pdb'):
        raise ValueError("File must be .xyz or .pdb")

    with open(filename, 'r') as f:
        mol_data = f.read()

    view = py3Dmol.view(width=width, height=height)

    # Add model
    view.addModel(mol_data, ext)

    # Style options
    if style == 'ballstick':
        view.setStyle({'stick': {'radius': 0.15},
                       'sphere': {'scale': 0.25}})
    elif style == 'stick':
        view.setStyle({'stick': {'radius': 0.15}})
    elif style == 'sphere':
        view.setStyle({'sphere': {'scale': 0.3}})
    else:
        view.setStyle({})  # raw atoms

    view.setBackgroundColor('white')
    view.zoomTo()
    return view.show()

# Example usage:
view_structure('../data/molecules/fe_co5.xyz')
# view_structure('/tmp/feco5_opt.xyz', style='stick')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:
# =============================================================================
# Minimal Molecular Viewer (py3Dmol)
# =============================================================================

import py3Dmol

def view_structure(filename, style='ballstick', width=500, height=400):
    """
    Minimal, clean py3Dmol viewer for XYZ or PDB files.
    Automatically detects format from file extension.
    """
    ext = filename.split('.')[-1].lower()
    if ext not in ('xyz', 'pdb'):
        raise ValueError("File must be .xyz or .pdb")

    with open(filename, 'r') as f:
        mol_data = f.read()

    view = py3Dmol.view(width=width, height=height)

    # Add model
    view.addModel(mol_data, ext)

    # Style options
    if style == 'ballstick':
        view.setStyle({'stick': {'radius': 0.15},
                       'sphere': {'scale': 0.25}})
    elif style == 'stick':
        view.setStyle({'stick': {'radius': 0.15}})
    elif style == 'sphere':
        view.setStyle({'sphere': {'scale': 0.3}})
    else:
        view.setStyle({})  # raw atoms

    view.setBackgroundColor('white')
    view.zoomTo()
    return view.show()

# Example usage:
view_structure('../data/molecules/ferrocene.xyz')
# view_structure('/tmp/feco5_opt.xyz', style='stick')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [1]:
# ------------------------------------------------------------------
# Parsing ORCA Output Files
# ------------------------------------------------------------------
import re

mock_orca_out = '''
ORCA SCF CONVERGED AFTER  11 CYCLES
Total Energy       :         -76.437879956 Eh
TOTAL ENERGY          -76.437879956 Hartree
TOTAL ENTHALPY        -76.435432876 Hartree

Mulliken charges:
   0 O   :  -0.708
   1 H   :   0.354
   2 H   :   0.354

VIBRATIONAL FREQUENCIES
   0:       0.00 cm**-1
   3:    1641.82 cm**-1
   4:    3779.56 cm**-1
   5:    3892.41 cm**-1
'''

e_match = re.search(r'TOTAL ENERGY\s+([\-0-9.]+)\s+Hartree', mock_orca_out)
charge_matches = re.findall(r'\s+\d+\s+(\w+)\s+:\s+([\-0-9.]+)', mock_orca_out)
freq_matches = re.findall(r'\s+\d+:\s+([0-9.]+) cm\*\*-1', mock_orca_out)

print('Parsed from mock ORCA .out:')
if e_match:
    print(f'  Total energy = {e_match.group(1)} Hartree')
print('  Mulliken charges:', {sym: float(q) for sym, q in charge_matches})
non_zero = [f for f in freq_matches if float(f) > 10]
print('  Vibrational frequencies (cm-1):', non_zero)

print()
print('--- cclib usage (with a real ORCA .out file) ---')
print('import cclib')
print('data = cclib.io.ccread("water.out")')
print('print(data.scfenergies[-1], "eV")')
print('print(data.atomcharges["mulliken"])')
print('print(data.vibfreqs, "cm-1")')

Parsed from mock ORCA .out:
  Total energy = -76.437879956 Hartree
  Mulliken charges: {'O': -0.708, 'H': 0.354}
  Vibrational frequencies (cm-1): ['1641.82', '3779.56', '3892.41']

--- cclib usage (with a real ORCA .out file) ---
import cclib
data = cclib.io.ccread("water.out")
print(data.scfenergies[-1], "eV")
print(data.atomcharges["mulliken"])
print(data.vibfreqs, "cm-1")


## 4. Common ORCA Keywords Reference

| Keyword | Purpose |
|---------|--------|
| `TightSCF` / `VeryTightSCF` | SCF convergence criteria |
| `Opt` | Geometry optimization |
| `Freq` | Analytical or numerical Hessian |
| `NMR` | NMR shielding tensors |
| `RIJCOSX` | RI-J Coulomb + seminumerical exchange (hybrids) |
| `RI` / `RIJONX` | RI approximation for pure DFT |
| `CPCM(Water)` | Conductor-like PCM solvation |
| `def2/J` | Auxiliary Coulomb-fitting basis |
| `def2-TZVP/C` | Auxiliary correlation basis for MP2/CC |
| `TightPNO` | Tight thresholds for DLPNO methods |
| `ZORA` / `DKH` | Scalar relativistic corrections |
| `D3BJ` | Grimme D3 dispersion with Becke-Johnson damping |
| `Grid5` / `Grid7` | Integration grid accuracy |

## Research Connection

ORCA is used across computational chemistry research:

- **Bioinorganic chemistry**: DLPNO-CCSD(T) energetics for iron-sulfur clusters and
  metalloenzyme active sites too large for canonical CC.
- **Catalysis**: BP86/TZVP geometry optimizations followed by DLPNO-CCSD(T) single-points
  is a standard protocol for reaction energy benchmarks.
- **Spectroscopy**: ORCA's EFG, EPR, and Mossbauer modules interpret transition metal spectra.
- **ML potentials**: ORCA generates training data for neural network potentials
  (e.g., wB97X-D3/def2-TZVP for ANI-2x).

## Summary

| Topic | Key Point |
|-------|----------|
| Input syntax | `! method basis [keywords]` + `* xyz charge mult ... *` |
| RIJCOSX | RI-J Coulomb + COSX exchange; ~10x faster hybrids |
| DLPNO-CCSD(T) | O(N^3) near-canonical CCSD(T); < 1 kcal/mol error |
| Auxiliary basis | `def2/J` for RI-J; `def2-TZVP/C` for correlation |
| Output parsing | cclib: `cclib.io.ccread('file.out')` |
| Parallelism | `%pal nprocs N end`; `%maxcore MB_per_core` |
| Grid | `Grid5` default; `Grid7` for final energies |

## Exercises

1. **Write an input**: Write an ORCA input for geometry optimization of ethanol
   (CH3CH2OH) using B3LYP-D3BJ/def2-TZVP with RIJCOSX. Include `%pal` and
   `%maxcore` blocks for 8 cores and 4 GB/core.

2. **DLPNO setup**: Write a DLPNO-CCSD(T)/def2-TZVP single-point input for
   the optimized ethanol geometry. Use TightPNO settings.

3. **cclib parsing**: Given a real ORCA frequency output, write Python code using
   cclib to (a) extract all vibrational frequencies, (b) compute the ZPE, and
   (c) identify imaginary modes.

4. **Basis set convergence**: Design an ORCA input series using def2-SVP, def2-TZVP,
   and def2-QZVP for HF single-points on the HF molecule.

5. **RIJCOSX accuracy**: Explain the RIJCOSX approximation. What property calculations
   might be sensitive to the COSX approximation for the exchange part?

! B3LYP D3BJ def2-TZVP def2/J RIJCOSX TightSCF Opt

%maxcore 4000
%pal
  nprocs 8
end

* xyz 0 1
  C     -1.200000    0.000000    0.000000
  C      0.300000    0.000000    0.000000
  O      1.000000    1.200000    0.000000
  H     -1.566000    1.027000    0.000000
  H     -1.566000   -0.513000    0.890000
  H     -1.566000   -0.513000   -0.890000
  H      0.666000   -1.027000    0.000000
  H      0.666000    0.513000    0.890000
  H      1.960000    1.200000    0.000000
*

In [2]:
# =============================================================================
# Run ORCA ethanol geometry optimization in single-core mode
# =============================================================================

import subprocess
from pathlib import Path

# ------------------------------------------------------------------
# 1. ORCA executable path
# ------------------------------------------------------------------

orca_exe = "/home/dawood/software/orca/orca"

# ------------------------------------------------------------------
# 2. File names
# ------------------------------------------------------------------

inp_file = Path("ethanol_opt_1core.inp")
out_file = Path("ethanol_opt_1core.out")

# ------------------------------------------------------------------
# 3. ORCA input
# ------------------------------------------------------------------
# Important change:
# - Removed %pal block
# - No nprocs 8
#
# This prevents ORCA from calling mpirun.

orca_input = """! B3LYP D3BJ def2-TZVP def2/J RIJCOSX TightSCF Opt

%maxcore 4000

* xyz 0 1
  C     -1.200000    0.000000    0.000000
  C      0.300000    0.000000    0.000000
  O      1.000000    1.200000    0.000000
  H     -1.566000    1.027000    0.000000
  H     -1.566000   -0.513000    0.890000
  H     -1.566000   -0.513000   -0.890000
  H      0.666000   -1.027000    0.000000
  H      0.666000    0.513000    0.890000
  H      1.960000    1.200000    0.000000
*
"""

# ------------------------------------------------------------------
# 4. Write input file
# ------------------------------------------------------------------

inp_file.write_text(orca_input)

print("Wrote ORCA input file:")
print(inp_file.resolve())

# ------------------------------------------------------------------
# 5. Run ORCA
# ------------------------------------------------------------------
# This line actually launches the quantum chemistry calculation.
# It is equivalent to running this in terminal:
# /home/dawood/software/orca/orca ethanol_opt_1core.inp > ethanol_opt_1core.out

with open(out_file, "w") as f:
    result = subprocess.run(
        [orca_exe, str(inp_file)],
        stdout=f,
        stderr=subprocess.PIPE,
        text=True
    )

# ------------------------------------------------------------------
# 6. Check command-level result
# ------------------------------------------------------------------

print("\nORCA process return code:", result.returncode)

if result.stderr.strip():
    print("\nORCA stderr:")
    print(result.stderr)

# ------------------------------------------------------------------
# 7. Check ORCA output file
# ------------------------------------------------------------------

text = out_file.read_text(errors="ignore")

print("\nOutput file:")
print(out_file.resolve())

print("\nOutput checks:")

if "ORCA TERMINATED NORMALLY" in text:
    print("  ORCA terminated normally.")
else:
    print("  ORCA did NOT terminate normally.")

if "THE OPTIMIZATION HAS CONVERGED" in text:
    print("  Geometry optimization converged.")
else:
    print("  Geometry optimization did NOT converge yet.")

print("\nLast 40 lines of output:")
print("-" * 80)
for line in text.splitlines()[-40:]:
    print(line)

Wrote ORCA input file:
/home/dawood/Ch121a-DFT/Module1_DFT/notebooks_v2/ethanol_opt_1core.inp

ORCA process return code: 0

Output file:
/home/dawood/Ch121a-DFT/Module1_DFT/notebooks_v2/ethanol_opt_1core.out

Output checks:
  ORCA terminated normally.
  Geometry optimization converged.

Last 40 lines of output:
--------------------------------------------------------------------------------
     Molec. Phys. 2012 110 , 2413-2417
     doi.org/10.1080/00268976.2012.687466
  3. Neese, F.
     The ORCA program system
     WIRES Comput. Molec. Sci. 2012 2(1), 73-78
     doi.org/10.1002/wcms.81
  4. Izsak, R.; Neese, F.; Klopper, W.
     Robust fitting techniques in the chain of spheres approximation to the Fock exchange: The role of the complementary space
     J. Chem. Phys. 2013 139 , 
     doi.org/10.1063/1.4819264
  5. Neese, F.
     Software update: the ORCA program system, version 4.0
     WIRES Comput. Molec. Sci. 2018 8(1), 1-6
     doi.org/10.1002/wcms.1327
  6. Neese, F.; Wennmohs

In [4]:
# =============================================================================
# Parse final ORCA energy from ethanol output
# =============================================================================

import re
from pathlib import Path

out_file = Path("ethanol_opt_1core.out")
text = out_file.read_text(errors="ignore")

# Find all FINAL SINGLE POINT ENERGY lines
energy_matches = re.findall(
    r"FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)",
    text
)

if energy_matches:
    final_energy = float(energy_matches[-1])
    print(f"Final electronic energy = {final_energy:.12f} Hartree")
else:
    print("No final energy found.")

Final electronic energy = -155.021322268442 Hartree


In [6]:
# =============================================================================
# Extract approximate optimized geometry from ORCA output
# =============================================================================

from pathlib import Path

out_file = Path("ethanol_opt_1core.out")
text = out_file.read_text(errors="ignore")
lines = text.splitlines()

# Find the last CARTESIAN COORDINATES block
block_starts = [
    i for i, line in enumerate(lines)
    if "CARTESIAN COORDINATES (ANGSTROEM)" in line
]

if not block_starts:
    print("No Cartesian coordinate block found.")
else:
    start = block_starts[-1]

    print("Last Cartesian coordinate block:")
    print("-" * 80)

    # Print about 20 lines after the header
    for line in lines[start:start + 20]:
        print(line)

Last Cartesian coordinate block:
--------------------------------------------------------------------------------
CARTESIAN COORDINATES (ANGSTROEM)
---------------------------------
  C     -1.150586    0.056860    0.032163
  C      0.335141    0.079413    0.320479
  O      0.926531    1.087188   -0.499527
  H     -1.598199    1.026203    0.257062
  H     -1.645295   -0.703442    0.640165
  H     -1.331997   -0.168651   -1.019979
  H      0.776943   -0.901488    0.103662
  H      0.508358    0.297300    1.382488
  H      1.873104    1.113617   -0.326513

----------------------------
CARTESIAN COORDINATES (A.U.)
----------------------------
  NO LB      ZA    FRAG     MASS         X           Y           Z
   0 C     6.0000    0    12.011   -2.174292    0.107450    0.060780
   1 C     6.0000    0    12.011    0.633324    0.150069    0.605618
   2 O     8.0000    0    15.999    1.750889    2.054487   -0.943969
   3 H     1.0000    0     1.008   -3.020158    1.939242    0.485776


The ethanol geometry optimization was successful. ORCA terminated normally and reported that the geometry optimization converged. The calculation used B3LYP-D3BJ/def2-TZVP with the RIJCOSX approximation and a def2/J auxiliary basis. The final electronic energy was -155.021322268442 Hartree. This absolute energy is not chemically meaningful by itself, but it can be compared to other ethanol conformers or related molecules calculated with the same method and basis set.

During the optimization, ORCA changed the initial Cartesian coordinates to reduce the molecular energy and forces. The final Cartesian coordinate block gives the optimized ethanol geometry in angstroms. The total wall time was about 4 minutes and 45 seconds. Most of the time was spent in SCF iterations and SCF gradient evaluations, which is expected for a DFT geometry optimization.

In [7]:
! DLPNO-CCSD(T) def2-TZVP def2-TZVP/C TightSCF TightPNO

%maxcore 4000

%pal
  nprocs 8
end

* xyzfile 0 1 ethanol_opt.xyz
    

IndentationError: unexpected indent (2041097643.py, line 6)

In [12]:
from pathlib import Path

xyz_file = Path("ethanol_opt.xyz")

print("Does ethanol_opt.xyz exist?", xyz_file.exists())

if xyz_file.exists():
    print()
    print(xyz_file.read_text())

Does ethanol_opt.xyz exist? True

9
Coordinates from ORCA-job ethanol_opt
  C          -1.20000000000000      0.00000000000000      0.00000000000000
  C           0.30000000000000      0.00000000000000      0.00000000000000
  O           1.00000000000000      1.20000000000000      0.00000000000000
  H          -1.56600000000000      1.02700000000000      0.00000000000000
  H          -1.56600000000000     -0.51300000000000      0.89000000000000
  H          -1.56600000000000     -0.51300000000000     -0.89000000000000
  H           0.66600000000000     -1.02700000000000      0.00000000000000
  H           0.66600000000000      0.51300000000000      0.89000000000000
  H           1.96000000000000      1.20000000000000      0.00000000000000



In [14]:
from pathlib import Path

# Read optimized ethanol geometry
xyz_lines = Path("ethanol_opt.xyz").read_text().splitlines()
geometry_lines = xyz_lines[2:]

# 1-core ORCA frequency input: no %pal block
orca_input = """! B3LYP D3BJ def2-TZVP def2/J RIJCOSX TightSCF Freq

%maxcore 4000

* xyz 0 1
""" + "\n".join(geometry_lines) + """
*
"""

Path("ethanol_freq_1core.inp").write_text(orca_input)

print("Wrote ethanol_freq_1core.inp")
print()
print(orca_input)

Wrote ethanol_freq_1core.inp

! B3LYP D3BJ def2-TZVP def2/J RIJCOSX TightSCF Freq

%maxcore 4000

* xyz 0 1
  C          -1.20000000000000      0.00000000000000      0.00000000000000
  C           0.30000000000000      0.00000000000000      0.00000000000000
  O           1.00000000000000      1.20000000000000      0.00000000000000
  H          -1.56600000000000      1.02700000000000      0.00000000000000
  H          -1.56600000000000     -0.51300000000000      0.89000000000000
  H          -1.56600000000000     -0.51300000000000     -0.89000000000000
  H           0.66600000000000     -1.02700000000000      0.00000000000000
  H           0.66600000000000      0.51300000000000      0.89000000000000
  H           1.96000000000000      1.20000000000000      0.00000000000000
*



In [15]:
import cclib
import numpy as np

filename = "ethanol_freq_1core.out"

data = cclib.io.ccread(filename)

freqs_cm = np.array(data.vibfreqs, dtype=float)

print("Number of frequencies:", len(freqs_cm))
print()

print("Vibrational frequencies:")
for i, freq in enumerate(freqs_cm, start=1):
    print(f"mode {i:3d}: {freq:10.2f} cm^-1")

[ORCA ethanol_freq_1core.out ERROR] Encountered error when parsing.
[ORCA ethanol_freq_1core.out ERROR] Last line read:   Last RMS-Density change    ...    3.0187e-06  Tolerance :   5.0000e-09



IndexError: list index out of range

In [17]:
from pathlib import Path

text = Path("ethanol_freq_1core.out").read_text(errors="ignore")

start = text.find("VIBRATIONAL FREQUENCIES")
end = text.find("NORMAL MODES")

freq_section = text[start:end]

print(freq_section)

VIBRATIONAL FREQUENCIES
-----------------------

Scaling factor for frequencies =  1.000000000  (already applied!)

     0:       0.00 cm**-1
     1:       0.00 cm**-1
     2:       0.00 cm**-1
     3:       0.00 cm**-1
     4:       0.00 cm**-1
     5:       0.00 cm**-1
     6:   -1091.06 cm**-1  ***imaginary mode***
     7:    -625.75 cm**-1  ***imaginary mode***
     8:    -422.67 cm**-1  ***imaginary mode***
     9:    -233.79 cm**-1  ***imaginary mode***
    10:     487.19 cm**-1
    11:     919.07 cm**-1
    12:     974.97 cm**-1
    13:    1001.10 cm**-1
    14:    1125.72 cm**-1
    15:    1176.21 cm**-1
    16:    1227.79 cm**-1
    17:    1387.24 cm**-1
    18:    1471.00 cm**-1
    19:    1476.20 cm**-1
    20:    1507.26 cm**-1
    21:    3020.46 cm**-1
    22:    3075.56 cm**-1
    23:    3077.38 cm**-1
    24:    3110.36 cm**-1
    25:    3218.26 cm**-1
    26:    3836.76 cm**-1


------------



In [18]:
import re
import numpy as np
from pathlib import Path

text = Path("ethanol_freq_1core.out").read_text(errors="ignore")

start = text.find("VIBRATIONAL FREQUENCIES")
end = text.find("NORMAL MODES")
freq_section = text[start:end]

matches = re.findall(r"^\s*\d+:\s*([\-0-9.]+)\s+cm\*\*-1", freq_section, flags=re.MULTILINE)

freqs_cm = np.array([float(x) for x in matches])

print("All frequencies:")
for i, freq in enumerate(freqs_cm, start=1):
    print(f"mode {i:3d}: {freq:10.2f} cm^-1")

print()
print("Imaginary frequencies:")
imaginary = freqs_cm[freqs_cm < 0]
print(imaginary)

All frequencies:
mode   1:       0.00 cm^-1
mode   2:       0.00 cm^-1
mode   3:       0.00 cm^-1
mode   4:       0.00 cm^-1
mode   5:       0.00 cm^-1
mode   6:       0.00 cm^-1
mode   7:   -1091.06 cm^-1
mode   8:    -625.75 cm^-1
mode   9:    -422.67 cm^-1
mode  10:    -233.79 cm^-1
mode  11:     487.19 cm^-1
mode  12:     919.07 cm^-1
mode  13:     974.97 cm^-1
mode  14:    1001.10 cm^-1
mode  15:    1125.72 cm^-1
mode  16:    1176.21 cm^-1
mode  17:    1227.79 cm^-1
mode  18:    1387.24 cm^-1
mode  19:    1471.00 cm^-1
mode  20:    1476.20 cm^-1
mode  21:    1507.26 cm^-1
mode  22:    3020.46 cm^-1
mode  23:    3075.56 cm^-1
mode  24:    3077.38 cm^-1
mode  25:    3110.36 cm^-1
mode  26:    3218.26 cm^-1
mode  27:    3836.76 cm^-1

Imaginary frequencies:
[-1091.06  -625.75  -422.67  -233.79]


In [19]:
positive_freqs_cm = freqs_cm[freqs_cm > 10.0]

cm_to_ev = 0.000123984198433
ev_to_hartree = 1.0 / 27.211386245988
hartree_to_kcal_mol = 627.509474

zpe_ev = 0.5 * positive_freqs_cm.sum() * cm_to_ev
zpe_hartree = zpe_ev * ev_to_hartree
zpe_kcal_mol = zpe_hartree * hartree_to_kcal_mol

print("Number of positive vibrational frequencies:", len(positive_freqs_cm))
print(f"ZPE = {zpe_ev:.8f} eV")
print(f"ZPE = {zpe_hartree:.10f} Hartree")
print(f"ZPE = {zpe_kcal_mol:.4f} kcal/mol")

Number of positive vibrational frequencies: 17
ZPE = 1.98948330 eV
ZPE = 0.0731121629 Hartree
ZPE = 45.8786 kcal/mol


The ORCA frequency output was parsed by extracting the VIBRATIONAL FREQUENCIES section with regex because cclib failed on this ORCA output format. The calculation gave 27 total frequency entries: 6 near-zero translational/rotational modes and 21 vibrational modes expected for ethanol.

For ethanol, N = 9 atoms, so a nonlinear molecule should have:

3N - 6 = 3(9) - 6 = 21 vibrational modes.

The output contains 4 imaginary vibrational modes:

-1091.06, -625.75, -422.67, and -233.79 cm^-1.

Therefore, this geometry is not a true local minimum. A true minimum should have zero imaginary frequencies. The ZPE computed from the positive frequencies only is:

ZPE = 1.98948330 eV
ZPE = 0.0731121629 Hartree
ZPE = 45.8786 kcal/mol

However, because the structure has imaginary modes, this ZPE should not be treated as a final physical thermochemical result. The geometry should be reoptimized before using the frequency calculation for final thermochemistry.

In [ ]:
! HF def2-SVP TightSCF

%maxcore 4000

%pal
  nprocs 4
end

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*

In [ ]:
! HF def2-TZVP TightSCF

%maxcore 4000

%pal
  nprocs 4
end

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*

In [ ]:
! HF def2-QZVP TightSCF

%maxcore 4000

%pal
  nprocs 4
end

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*

In [24]:
# ------------------------------------------------------------------
# Create ORCA input file: HF molecule, HF/def2-SVP single point
# ------------------------------------------------------------------

hf_def2_svp_input = """! HF def2-SVP TightSCF

%maxcore 4000

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*
"""

with open("hf_def2_svp.inp", "w") as f:
    f.write(hf_def2_svp_input)

print("Wrote hf_def2_svp.inp")

Wrote hf_def2_svp.inp


In [25]:
!ls -lh hf_def2_svp.inp
!cat hf_def2_svp.inp

-rw-r--r-- 1 dawood dawood 135 May  7 04:28 hf_def2_svp.inp
! HF def2-SVP TightSCF

%maxcore 4000

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*


In [26]:
# ------------------------------------------------------------------
# Create ORCA input files:
# HF molecule, Hartree-Fock single points with larger basis sets
# ------------------------------------------------------------------

hf_def2_tzvp_input = """! HF def2-TZVP TightSCF

%maxcore 4000

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*
"""

hf_def2_qzvp_input = """! HF def2-QZVP TightSCF

%maxcore 4000

* xyz 0 1
  H      0.000000    0.000000    0.000000
  F      0.000000    0.000000    0.917000
*
"""

with open("hf_def2_tzvp.inp", "w") as f:
    f.write(hf_def2_tzvp_input)

with open("hf_def2_qzvp.inp", "w") as f:
    f.write(hf_def2_qzvp_input)

print("Wrote hf_def2_tzvp.inp")
print("Wrote hf_def2_qzvp.inp")

Wrote hf_def2_tzvp.inp
Wrote hf_def2_qzvp.inp


In [27]:
!grep "FINAL SINGLE POINT ENERGY" hf_def2_svp.out
!grep "FINAL SINGLE POINT ENERGY" hf_def2_tzvp.out
!grep "FINAL SINGLE POINT ENERGY" hf_def2_qzvp.out


FINAL SINGLE POINT ENERGY       -99.932493507327
FINAL SINGLE POINT ENERGY      -100.063316081729
FINAL SINGLE POINT ENERGY      -100.070193306804


## RIJCOSX Accuracy

RIJCOSX is an approximation used in ORCA to speed up hybrid DFT calculations.

Hybrid DFT functionals, such as B3LYP or PBE0, include some Hartree-Fock exact exchange. Exact exchange is expensive because it requires many electron-electron repulsion integrals. RIJCOSX makes this faster by approximating two parts of the calculation.

The `RI-J` part approximates the Coulomb contribution using an auxiliary basis set.

The `COSX` part approximates the exact-exchange contribution using a semi-numerical grid method called chain-of-spheres exchange.

So the name means:

$$
\text{RIJCOSX} = \text{RI-J Coulomb approximation} + \text{COSX exchange approximation}
$$

The Coulomb part describes the average electrostatic repulsion between electrons. The exchange part comes from the antisymmetry of the electronic wave function, meaning electrons with the same spin cannot occupy the same quantum state.

In practical terms, RIJCOSX is usually safe for routine hybrid DFT geometry optimizations and single-point energies. It often gives nearly the same results as the full hybrid DFT calculation, but much faster.

A typical ORCA input line is:

```orca
! B3LYP D3BJ def2-TZVP def2/J RIJCOSX TightSCF Opt

---

# 13 -- Jaguar: Capabilities & Practical Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/13_jaguar_tutorial.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Jaguar `.in` input file format (sections and keywords)
- Set up DFT, LMP2, and GVB-PP calculations in Jaguar
- Use the Maestro GUI workflow for job submission and visualization
- Parse Jaguar `.out` files with Python and cclib
- Understand Jaguar's strengths: LMP2, GVB, transition metal DFT
- Know when to choose Jaguar vs ORCA vs PySCF

## 1. Jaguar Overview

**Jaguar** (Schrodinger, Inc.) is a commercial quantum chemistry code optimized for:

| Capability | Details |
|------------|--------|
| DFT | All major XC functionals; pseudospectral method for speed |
| LMP2 | Local MP2 -- scales O(N^3) vs O(N^5) for canonical MP2 |
| GVB-PP | Generalized Valence Bond (perfect pairing) for static correlation |
| Transition metals | LACVP**, ECP basis sets built-in |
| Solvation | PBF (Poisson-Boltzmann Finite element) solvent model |
| Integration | Schrodinger Suite: Maestro, Glide, Prime, MacroModel |

**License**: Commercial. Academic licenses available through Schrodinger.

## 2. Input File Format

A Jaguar input file (`job.in`) uses named sections delimited by `&section ... &`:

### 2.1 Minimal Single-Point

```
&gen
 basis=6-31g**
 dftname=b3lyp
 igeopt=0
 molchg=0
 multip=1
&

&zmat
 O  0.000  0.000  0.117
 H  0.000  0.757 -0.469
 H  0.000 -0.757 -0.469
&
```

### 2.2 Geometry Optimization

```
&gen
 basis=def2-tzvp
 dftname=b3lyp
 igeopt=1
 maxitg=100
 gconv=5.0e-5
&
```

### 2.3 LMP2 Single-Point

```
&gen
 basis=cc-pvtz
 mp2=yes
 lmp2=yes
 igeopt=0
&
```

### 2.4 Transition Metal (LACVP** ECP)

```
&gen
 basis=lacvp**
 dftname=b3lyp
 igeopt=1
 molchg=0
 multip=1
&

&zmat
 Fe  0.000  0.000  0.000
 C   0.000  0.000  1.807
 O   0.000  0.000  2.975
 ...
&
```

In [3]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT -- Notebook 13: Jaguar Tutorial
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================

# ------------------------------------------------------------------
# Writing Jaguar Input Files with Python
# ------------------------------------------------------------------

def write_jaguar_input(filename, atoms, basis='6-31g**', dftname='b3lyp',
                       igeopt=0, charge=0, mult=1, extra_gen='', extra_sections=''):
    '''Write a Jaguar .in file.'''
    gen = f'basis={basis}\ndftname={dftname}\nigeopt={igeopt}\nmolchg={charge}\nmultip={mult}'
    if extra_gen:
        gen += '\n' + extra_gen
    zmat = '\n'.join(
        f' {sym:2s}  {x:10.5f}  {y:10.5f}  {z:10.5f}'
        for sym, x, y, z in atoms
    )
    content = f'&gen\n{gen}\n&\n\n&zmat\n{zmat}\n&\n'
    if extra_sections:
        content += '\n' + extra_sections
    with open(filename, 'w') as f:
        f.write(content)
    return content

water = [('O',0.000,0.000,0.117),('H',0.000,0.757,-0.469),('H',0.000,-0.757,-0.469)]

print('=== B3LYP/def2-TZVP single-point ===')
print(write_jaguar_input('/tmp/water_sp.in', water, basis='def2-tzvp', dftname='b3lyp'))

print('=== LMP2/cc-pVTZ single-point ===')
print(write_jaguar_input('/tmp/water_lmp2.in', water,
                         basis='cc-pvtz', dftname='none',
                         extra_gen='mp2=yes\nlmp2=yes'))

print('=== B3LYP/6-31G** + PBF solvation (water) ===')
print(write_jaguar_input('/tmp/water_pbf.in', water,
                         extra_gen='isolv=2\nsolvent=water'))

=== B3LYP/def2-TZVP single-point ===
&gen
basis=def2-tzvp
dftname=b3lyp
igeopt=0
molchg=0
multip=1
&

&zmat
 O      0.00000     0.00000     0.11700
 H      0.00000     0.75700    -0.46900
 H      0.00000    -0.75700    -0.46900
&

=== LMP2/cc-pVTZ single-point ===
&gen
basis=cc-pvtz
dftname=none
igeopt=0
molchg=0
multip=1
mp2=yes
lmp2=yes
&

&zmat
 O      0.00000     0.00000     0.11700
 H      0.00000     0.75700    -0.46900
 H      0.00000    -0.75700    -0.46900
&

=== B3LYP/6-31G** + PBF solvation (water) ===
&gen
basis=6-31g**
dftname=b3lyp
igeopt=0
molchg=0
multip=1
isolv=2
solvent=water
&

&zmat
 O      0.00000     0.00000     0.11700
 H      0.00000     0.75700    -0.46900
 H      0.00000    -0.75700    -0.46900
&



## 3. GVB-PP: Generalized Valence Bond

**GVB (Generalized Valence Bond)** describes electron pairs as geminal products
of two natural orbitals, capturing static (near-degeneracy) correlation that DFT misses.

### GVB-PP Wavefunction

$$|\Psi_{\rm GVB}\rangle = \prod_{i=1}^{N/2}
(c_{i1}\phi_{i1} + c_{i2}\phi_{i2})_\alpha
(c_{i1}\phi_{i1} + c_{i2}\phi_{i2})_\beta$$

**When to use GVB**:
- Bond-breaking reactions (where DFT and RHF break down)
- Diradical / biradical character
- Homolytic bond dissociation curves
- Starting orbitals for CASSCF / MRCI

### GVB Input Example

```
&gen
 basis=6-31g**
 igvb=1
 npair=2
 igeopt=0
&
```

`npair`: number of GVB pairs to correlate (start with the bonds of interest).

In [4]:
# ------------------------------------------------------------------
# Parsing Jaguar Output Files with Python / cclib
# ------------------------------------------------------------------
import re

mock_jaguar_out = '''
  Jaguar version 11.3, release 011

  Total energy:        -76.437819  hartrees

  ATOMIC COORDINATES
   Atom       X          Y          Z
     O     0.000000   0.000000   0.117000
     H     0.000000   0.757000  -0.469000
     H     0.000000  -0.757000  -0.469000

  Mulliken charges:
     O    -0.7024
     H     0.3512
     H     0.3512

  Vibrational frequencies:
    Mode 1:   1643.2 cm-1
    Mode 2:   3802.4 cm-1
    Mode 3:   3919.1 cm-1

  ZPE           =   0.021143 hartrees
  Total enthalpy = -76.410843 hartrees
'''

e_match  = re.search(r'Total energy:\s+([\-0-9.]+)\s+hartrees', mock_jaguar_out)
charges  = re.findall(r'(O|H|C|N|F|Cl)\s+([\-0-9.]+)', mock_jaguar_out)
freqs    = re.findall(r'Mode \d+:\s+([0-9.]+) cm-1', mock_jaguar_out)
zpe      = re.search(r'ZPE\s+=\s+([0-9.]+) hartrees', mock_jaguar_out)

HARTREE2KCAL = 627.5095
print('Parsed from mock Jaguar .out:')
if e_match:
    print(f'  Total energy = {e_match.group(1)} Hartree')
print(f'  Mulliken charges: { {s: float(q) for s, q in charges} }')
print(f'  Frequencies: {freqs} cm-1')
if zpe:
    zpe_kcal = float(zpe.group(1)) * HARTREE2KCAL
    print(f'  ZPE = {zpe.group(1)} Hartree = {zpe_kcal:.2f} kcal/mol')

print()
print('--- cclib usage (with a real Jaguar .out file) ---')
print('import cclib')
print('data = cclib.io.ccread("job.out")')
print('print(data.scfenergies)   # SCF energies in eV')
print('print(data.atomcharges)   # Mulliken / Lowdin charges')
print('print(data.vibfreqs)      # Vibrational frequencies')
print('print(data.zpve)          # Zero-point vibrational energy')

Parsed from mock Jaguar .out:
  Total energy = -76.437819 Hartree
  Mulliken charges: {'O': -0.7024, 'H': 0.3512}
  Frequencies: ['1643.2', '3802.4', '3919.1'] cm-1
  ZPE = 0.021143 Hartree = 13.27 kcal/mol

--- cclib usage (with a real Jaguar .out file) ---
import cclib
data = cclib.io.ccread("job.out")
print(data.scfenergies)   # SCF energies in eV
print(data.atomcharges)   # Mulliken / Lowdin charges
print(data.vibfreqs)      # Vibrational frequencies
print(data.zpve)          # Zero-point vibrational energy


## 4. Jaguar vs ORCA vs PySCF

| Feature | PySCF | ORCA | Jaguar |
|---------|-------|------|--------|
| License | Open-source | Free academic | Commercial |
| DFT | Yes | Yes (faster) | Yes (pseudospectral) |
| LMP2 | -- | DLPNO-MP2 | Yes (native) |
| GVB | -- | -- | Yes (native) |
| DLPNO-CCSD(T) | -- | Yes | -- |
| Transition metals | Yes | Yes | Yes (LACVP** built-in) |
| Solvation | ddCOSMO | CPCM | PBF |
| GUI | -- | -- | Maestro |
| Python API | Native | cclib | cclib |

**Use Jaguar when**: LMP2 accuracy is needed for large systems, GVB for
multireference character, or the Maestro GUI workflow is required.

## 5. Maestro GUI Workflow

The **Maestro** GUI (Schrodinger Suite) provides a graphical interface for Jaguar:

1. **Build / Import**: Draw or import structure into Maestro Project Table
2. **Jaguar panel**: `Applications -> Quantum Mechanics -> Jaguar`
3. **Theory tab**: Choose DFT functional, basis set, charge, multiplicity
4. **Properties tab**: Check: energy, charges, frequencies, NMR
5. **Solvation tab**: Choose PBF solvent (water, chloroform, octanol...)
6. **Submit**: Run locally or on cluster via Job Control
7. **Analyze**: View optimized structure, vibrations, MOs in Maestro

**Project Table**: Tracks all jobs, properties, and structures in one place.
Especially useful for series of related calculations (conformer searches, pKa, etc.).

## Research Connection

Jaguar is used in pharmaceutical and materials research workflows:

- **Drug discovery**: Jaguar pKa prediction module combines GVB + PBF solvation;
  used in large-scale virtual screening at pharmaceutical companies.
- **Transition metal catalysis**: LACVP** built-in ECPs simplify setup for Pd, Rh,
  and Ir-catalyzed reactions in natural product synthesis planning.
- **Conformer energetics**: Fast DFT/LMP2 relative energies for OPLS force field
  parameterisation (protein-ligand binding free energy calculations).
- **Materials**: Jaguar embedded in Schrodinger Materials Science Suite for
  electronic structure of organic semiconductors and battery electrolytes.

## Summary

| Topic | Key Point |
|-------|----------|
| Input format | `&gen ... &` + `&zmat ... &` sections |
| DFT keyword | `dftname=b3lyp`, `basis=def2-tzvp` |
| Opt keyword | `igeopt=1` |
| LMP2 | `mp2=yes`, `lmp2=yes`; O(N^3) scaling |
| GVB-PP | `igvb=1`, `npair=N`; static correlation |
| TM basis | `lacvp**` (ECP for 1st/2nd-row transition metals) |
| PBF solvation | `isolv=2`, `solvent=water` |
| cclib parsing | `cclib.io.ccread('job.out')` |
| GUI | Maestro -> Quantum Mechanics -> Jaguar |

## Exercises

1. **Write Jaguar inputs**: Write `.in` files for (a) a B3LYP/6-31G** geometry
   optimization of methanol and (b) an LMP2/cc-pVTZ single-point on the result.

2. **GVB for bond breaking**: Explain why GVB-PP gives a better description of
   the H2 dissociation curve than RHF or DFT at large bond distances.

3. **LMP2 vs DLPNO-MP2**: Compare the LMP2 (Jaguar) and DLPNO-MP2 (ORCA) approaches.
   Both achieve O(N^3) scaling -- what approximations do they each make?

4. **LACVP** basis**: Look up the LACVP** basis set. Which atoms use ECPs and which
   use all-electron basis functions? Why are ECPs useful for heavy transition metals?

5. **Workflow comparison**: You need to compute the binding free energy of a Pd(II)
   complex with a phosphine ligand. Compare how you would set this up in PySCF,
   ORCA, and Jaguar. What are the pros and cons of each?

In [5]:
# ------------------------------------------------------------------
# Exercise: Write Jaguar inputs for methanol
# ------------------------------------------------------------------

def write_jaguar_input(filename, atoms, basis='6-31g**', dftname='b3lyp',
                       igeopt=0, charge=0, mult=1,
                       extra_gen='', extra_sections=''):
    """
    Write a Jaguar .in file.

    Parameters
    ----------
    filename : str
        Output filename, such as '/tmp/methanol_opt.in'.

    atoms : list of tuples
        Each tuple has the form:
        ('Element', x, y, z)

    basis : str
        Basis set, such as '6-31g**' or 'cc-pvtz'.

    dftname : str
        DFT functional name, such as 'b3lyp'.
        Use 'none' if doing MP2 instead of DFT.

    igeopt : int
        0 = single-point calculation.
        1 = geometry optimization.

    charge : int
        Molecular charge.

    mult : int
        Spin multiplicity.
        1 = singlet, 2 = doublet, 3 = triplet.

    extra_gen : str
        Extra lines to add inside the &gen section.

    extra_sections : str
        Extra Jaguar sections to add after &zmat.
    """

    # Build the &gen settings block
    gen = (
        f'basis={basis}\n'
        f'dftname={dftname}\n'
        f'igeopt={igeopt}\n'
        f'molchg={charge}\n'
        f'multip={mult}'
    )

    # Add optional extra settings
    if extra_gen:
        gen += '\n' + extra_gen

    # Format the molecular coordinates
    zmat = '\n'.join(
        f' {sym:2s}  {x:10.5f}  {y:10.5f}  {z:10.5f}'
        for sym, x, y, z in atoms
    )

    # Combine everything into Jaguar input format
    content = f'&gen\n{gen}\n&\n\n&zmat\n{zmat}\n&\n'

    # Add optional extra sections
    if extra_sections:
        content += '\n' + extra_sections

    # Write the file
    with open(filename, 'w') as f:
        f.write(content)

    return content


# ------------------------------------------------------------------
# Approximate starting geometry for methanol, CH3OH
# ------------------------------------------------------------------
# This is only a reasonable starting geometry.
# Jaguar will optimize it in part (a).
methanol = [
    ('C',   0.00000,   0.00000,   0.00000),
    ('O',   1.43000,   0.00000,   0.00000),
    ('H',  -0.36000,   1.02000,   0.00000),
    ('H',  -0.36000,  -0.51000,   0.88300),
    ('H',  -0.36000,  -0.51000,  -0.88300),
    ('H',   1.79000,   0.89000,   0.00000),
]


# ------------------------------------------------------------------
# (a) B3LYP/6-31G** geometry optimization of methanol
# ------------------------------------------------------------------
methanol_opt_input = write_jaguar_input(
    filename='/tmp/methanol_b3lyp_631gss_opt.in',
    atoms=methanol,
    basis='6-31g**',
    dftname='b3lyp',
    igeopt=1,
    charge=0,
    mult=1,
    extra_gen='maxitg=100\ngconv=5.0e-5'
)

print('=== (a) Methanol B3LYP/6-31G** geometry optimization ===')
print(methanol_opt_input)


# ------------------------------------------------------------------
# (b) LMP2/cc-pVTZ single-point on the optimized result
# ------------------------------------------------------------------
# In real use, replace the coordinates below with the optimized geometry
# from the Jaguar output of part (a).
methanol_lmp2_input = write_jaguar_input(
    filename='/tmp/methanol_lmp2_ccpvtz_sp.in',
    atoms=methanol,
    basis='cc-pvtz',
    dftname='none',
    igeopt=0,
    charge=0,
    mult=1,
    extra_gen='mp2=yes\nlmp2=yes'
)

print('=== (b) Methanol LMP2/cc-pVTZ single-point ===')
print(methanol_lmp2_input)

=== (a) Methanol B3LYP/6-31G** geometry optimization ===
&gen
basis=6-31g**
dftname=b3lyp
igeopt=1
molchg=0
multip=1
maxitg=100
gconv=5.0e-5
&

&zmat
 C      0.00000     0.00000     0.00000
 O      1.43000     0.00000     0.00000
 H     -0.36000     1.02000     0.00000
 H     -0.36000    -0.51000     0.88300
 H     -0.36000    -0.51000    -0.88300
 H      1.79000     0.89000     0.00000
&

=== (b) Methanol LMP2/cc-pVTZ single-point ===
&gen
basis=cc-pvtz
dftname=none
igeopt=0
molchg=0
multip=1
mp2=yes
lmp2=yes
&

&zmat
 C      0.00000     0.00000     0.00000
 O      1.43000     0.00000     0.00000
 H     -0.36000     1.02000     0.00000
 H     -0.36000    -0.51000     0.88300
 H     -0.36000    -0.51000    -0.88300
 H      1.79000     0.89000     0.00000
&



The first file is a B3LYP/6-31G** geometry optimization because dftname=b3lyp selects the DFT functional, basis=6-31g** selects the basis set, and igeopt=1 tells Jaguar to move the atoms until it finds a lower-energy methanol structure. The second file is an LMP2/cc-pVTZ single-point because mp2=yes and lmp2=yes turn on local MP2, basis=cc-pvtz uses a larger correlation-consistent basis, and igeopt=0 keeps the geometry fixed.

Explain why GVB-PP gives a better description of the H2 dissociation curve than RHF or DFT at large bond distances

GVB-PP gives a better description of the $\mathrm{H_2}$ dissociation curve because it can describe **static correlation** during bond breaking.

At equilibrium, $\mathrm{H_2}$ is mostly well described by a closed-shell electron pair in the bonding orbital:

$$
\sigma_g^2
$$

So RHF is reasonable near the equilibrium bond length.

But as the H-H bond is stretched, the correct dissociation limit is:

$$
\mathrm{H_2 \rightarrow H\cdot + H\cdot}
$$

That means each hydrogen atom should have one electron. This requires static correlation.

Static correlation means that more than one electron configuration becomes important. This happens when bonds break, when diradicals form, or when orbitals become nearly degenerate.

RHF fails because it forces both electrons to stay paired in the same spatial orbital. At large H-H distance, the RHF bonding orbital is approximately:

$$
\sigma_g = \frac{1}{\sqrt{2}}(1s_A + 1s_B)
$$

So the RHF wavefunction contains:

$$
\sigma_g^2
=
\frac{1}{2}(1s_A + 1s_B)^2
$$

Expanding this gives covalent terms and ionic terms:

$$
\sigma_g^2
=
\frac{1}{2}
\left[
1s_A(1)1s_A(2)
+
1s_A(1)1s_B(2)
+
1s_B(1)1s_A(2)
+
1s_B(1)1s_B(2)
\right]
$$

The middle two terms are the desired covalent terms:

$$
\mathrm{H\cdot + H\cdot}
$$

But the first and last terms are ionic terms:

$$
\mathrm{H^- + H^+}
$$

and

$$
\mathrm{H^+ + H^-}
$$

These ionic terms are unphysical at infinite separation. Two separated hydrogen atoms should be neutral radicals, not a mixture containing ionic fragments.

GVB-PP fixes the main problem by letting the bonding electron pair use two correlated orbitals instead of one doubly occupied orbital. As the H-H bond stretches, these orbitals can localize on different hydrogen atoms:

$$
\phi_A \approx 1s_A
$$

$$
\phi_B \approx 1s_B
$$

So the GVB-PP wavefunction gives the correct large-distance picture:

$$
\mathrm{one\ electron\ on\ H_A}
$$

$$
\mathrm{one\ electron\ on\ H_B}
$$

In molecular orbital language, GVB-PP effectively mixes the bonding and antibonding configurations:

$$
\sigma_g^2
\quad \text{and} \quad
\sigma_u^2
$$

The antibonding configuration helps cancel the bad ionic terms in the RHF wavefunction.

Therefore, GVB-PP gives a better $\mathrm{H_2}$ dissociation curve because it captures the left-right/static correlation needed for bond breaking. RHF keeps an unphysical closed-shell ionic mixture at large bond distances. Unrestricted HF or unrestricted DFT can improve the energy, but they often do this by breaking spin symmetry. GVB-PP gives the correct qualitative dissociation picture without relying on artificial spin-symmetry breaking.



I used an LLM for formatting my answer. explanation was inspired from A Chemist guide to DFT by Koch


Both Jaguar LMP2 and ORCA DLPNO-MP2 reduce the cost of canonical MP2 by using locality. Canonical MP2 correlates every occupied orbital pair with every virtual orbital pair, which gives roughly O(N^5) scaling. This is wasteful for large molecules because electron correlation is mostly local: a bond or lone pair is most strongly correlated with nearby virtual orbitals.

Jaguar LMP2 uses localized occupied orbitals and restricts the virtual space assigned to each localized orbital or occupied pair. In simple terms, each bond or lone pair is only allowed to correlate with nearby virtual orbitals rather than with the entire virtual orbital space of the molecule. This reduces the scaling and makes MP2 practical for larger systems. In the limit where all local virtual orbitals are assigned to every occupied orbital, Jaguar LMP2 becomes equivalent to canonical MP2.

ORCA DLPNO-MP2 uses a domain-based local pair natural orbital approximation. It also starts from localized occupied orbitals, but for each occupied pair it builds a local domain and then compresses the virtual space into pair natural orbitals. Only the pair natural orbitals with significant occupations are retained. Thus, DLPNO-MP2 makes two main approximations: it restricts correlation to local pair domains and truncates the pair-specific virtual space using PNO thresholds.

The key difference is that Jaguar LMP2 mainly uses localized occupied orbitals plus local virtual spaces, while ORCA DLPNO-MP2 uses localized occupied pairs, domain restrictions, and pair natural orbital compression. Both are based on the same physical idea that electron correlation is local, but DLPNO-MP2 has a more explicit pair-specific virtual-space compression.

LACVP** uses normal all-electron basis functions for light atoms like H, C, N, O, F, P, S, and Cl, but uses effective core potentials for heavier atoms such as transition metals. The ECP replaces inner core electrons with an effective potential, so the calculation focuses on the chemically important valence electrons. This makes heavy-metal calculations cheaper and often more stable, while also including important scalar relativistic effects.


for pyscf

1. Build geometries for:
   - Pd fragment
   - phosphine ligand
   - full Pd-phosphine complex

2. Run DFT geometry optimizations.
   Example method:
   PBE0-D3 or B3LYP-D3
   basis: def2-SVP or def2-TZVP
   ECP: def2-ECP for Pd

3. Run frequency calculations if available through your workflow.
   In pure PySCF this is less beginner-friendly than in ORCA/Jaguar.

4. Add solvent with ddCOSMO/ddPCM if needed.

5. Compute:
   ΔGbind = Gcomplex - Gfragment - Gligand

for orca

1. Generate starting structures:
   Pd fragment
   phosphine ligand
   full complex

2. Optimize each structure:
   PBE0-D3BJ/def2-SVP or B3LYP-D3BJ/def2-SVP
   RIJCOSX def2/J
   CPCM(solvent)

3. Frequency calculation on each optimized structure:
   confirm no imaginary frequencies
   obtain thermal free energy corrections

4. Single-point refinement:
   def2-TZVP or def2-QZVP
   maybe DLPNO-CCSD(T) if system size allows

5. Compute:
   ΔGbind = Gcomplex - Gfragment - Gligand + standard-state correction

for jaguar

1. Build Pd complex and fragments in Maestro.

2. Choose Jaguar calculation:
   B3LYP or another DFT functional
   LACVP** or larger basis
   PBF solvent model

3. Optimize:
   complex
   Pd fragment
   phosphine ligand

4. Run frequencies:
   get thermal corrections
   check imaginary modes

5. Optionally refine:
   LMP2 single-points if appropriate
   or use Jaguar-specific workflows for ligand/protonation/solvation problems

6. Compute:
   ΔGbind = Gcomplex - Gfragment - Gligand + standard-state correction

| Tool | Pros | Cons |
|---|---|---|
| PySCF | Open-source; native Python; best for scripting and learning theory; easy to automate many calculations | Less beginner-friendly for full workflows; geometry optimization, frequencies, and thermochemistry take more setup; transition-metal jobs require more manual choices |
| ORCA | Strong general-purpose quantum chemistry code; great for DFT, transition metals, frequencies, spectroscopy, and DLPNO-CCSD(T); free for academics | Mostly terminal/input-file based; no full GUI; binding free energy still needs careful bookkeeping |
| Jaguar | Maestro GUI makes setup and visualization easy; strong for Schrödinger workflows; useful LMP2, GVB-PP, PBF solvation, and LACVP** support | Commercial license; less open and scriptable; less transparent for learning theory; no ORCA-style DLPNO-CCSD(T) |